# nanochat base training notebook

This notebook is a notebook-first wrapper around `scripts/base_train.py`: configure Python variables below (no shell CLI required), then run training.

## 1) Configure training
Set `training_tokens` and other hyperparameters here. The notebook will convert this into the equivalent base-train arguments.

In [1]:
from math import ceil

TRAINING_CONFIG = {
    # Logging/runtime
    'run': 'nb-base-train',
    'data_dir': None,
    'checkpoints_dir': None,
    'device_type': '',  # ''=autodetect, or set 'cuda'/'cpu'/'mps'

    # Model
    'depth': 4, #20,
    'aspect_ratio': 64,
    'head_dim': 16, #128,
    'max_seq_len': 32,#2048,
    'window_pattern': 'SSSL',

    # Batch + training horizon
    'device_batch_size': 8,
    'total_batch_size': 524288//64,  # in tokens
    'training_tokens': 20 * (524288//64),  # <-- edit this to control total training budget

    # Optimization
    'embedding_lr': 0.3,
    'unembedding_lr': 0.004,
    'weight_decay': 0.2,
    'matrix_lr': 0.02,
    'scalar_lr': 0.5,
    'adam_beta1': 0.8,
    'adam_beta2': 0.95,
    'warmup_ratio': 0.0,
    'warmdown_ratio': 0.5,
    'final_lr_frac': 0.0,

    # Evaluation/checkpoint cadence
    'eval_every': 20,#250,
    'eval_tokens': 2 * 524288,
    'core_metric_every': 20,
    'core_metric_max_per_task': 500,
    'sample_every': 500,
    'save_every': 500,

    # Extras
    'resume_from_step': -1,
    'model_tag': 'nb_d20',
    'fp8': False,
    'fp8_recipe': 'tensorwise',
}

TRAINING_CONFIG['num_iterations'] = ceil(TRAINING_CONFIG['training_tokens'] / TRAINING_CONFIG['total_batch_size'])
print('Planned iterations:', TRAINING_CONFIG['num_iterations'])
print('Planned tokens   :', TRAINING_CONFIG['num_iterations'] * TRAINING_CONFIG['total_batch_size'])

Planned iterations: 20
Planned tokens   : 163840


## 2) Build argv in Python (no command-line typing)
This converts notebook config values into the same flags expected by `scripts/base_train.py`.

In [2]:
def config_to_argv(cfg: dict) -> list[str]:
    flag_map = {
        'run': '--run',
        'device_type': '--device-type',
        'data_dir': '--data-dir',
        'checkpoints_dir': '--checkpoints-dir',
        'depth': '--depth',
        'aspect_ratio': '--aspect-ratio',
        'head_dim': '--head-dim',
        'max_seq_len': '--max-seq-len',
        'window_pattern': '--window-pattern',
        'num_iterations': '--num-iterations',
        'device_batch_size': '--device-batch-size',
        'total_batch_size': '--total-batch-size',
        'embedding_lr': '--embedding-lr',
        'unembedding_lr': '--unembedding-lr',
        'weight_decay': '--weight-decay',
        'matrix_lr': '--matrix-lr',
        'scalar_lr': '--scalar-lr',
        'adam_beta1': '--adam-beta1',
        'adam_beta2': '--adam-beta2',
        'warmup_ratio': '--warmup-ratio',
        'warmdown_ratio': '--warmdown-ratio',
        'final_lr_frac': '--final-lr-frac',
        'resume_from_step': '--resume-from-step',
        'eval_every': '--eval-every',
        'eval_tokens': '--eval-tokens',
        'core_metric_every': '--core-metric-every',
        'core_metric_max_per_task': '--core-metric-max-per-task',
        'sample_every': '--sample-every',
        'save_every': '--save-every',
        'model_tag': '--model-tag',
        'fp8_recipe': '--fp8-recipe',
    }

    argv = ['scripts.base_train']
    if cfg.get('fp8', False):
        argv.append('--fp8')

    for k, flag in flag_map.items():
        if k not in cfg:
            continue
        argv.extend([flag, str(cfg[k])])
    return argv

argv = config_to_argv(TRAINING_CONFIG)
print('Generated argv:')
print(' '.join(argv[:12]), '...')

Generated argv:
scripts.base_train --run nb-base-train --device-type  --depth 4 --aspect-ratio 64 --head-dim 16 --max-seq-len ...


## 3) Launch training from notebook
Runs `scripts.base_train` in-process using the generated argv.

In [3]:
# !pip install --upgrade wandb


In [4]:
!wandb login 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


wandb: [wandb.login()] Using explicit session credentials for https://api.wandb.ai.
wandb: Appending key for api.wandb.ai to your netrc file: /home/seqaeon/.netrc
wandb: W&B API key is configured. Use `wandb login --relogin` to force relogin


In [5]:
import runpy
import sys

old_argv = sys.argv[:]
sys.argv = argv
try:
    runpy.run_module('scripts.base_train', run_name='__main__')
finally:
    sys.argv = old_argv

2026-03-08 18:31:48,208 - nanochat.common - INFO - Distributed world size: 1
2026-03-08 18:31:48,209 - nanochat.common - WARNING - Peak flops undefined for: NVIDIA GeForce RTX 3050 Ti Laptop GPU, MFU will show as 0%
wandb: [wandb.login()] Loaded credentials for https://api.wandb.ai from /home/seqaeon/.netrc.



                                                       █████                █████
                                                      ░░███                ░░███
     ████████    ██████   ████████    ██████   ██████  ░███████    ██████  ███████
    ░░███░░███  ░░░░░███ ░░███░░███  ███░░███ ███░░███ ░███░░███  ░░░░░███░░░███░
     ░███ ░███   ███████  ░███ ░███ ░███ ░███░███ ░░░  ░███ ░███   ███████  ░███
     ░███ ░███  ███░░███  ░███ ░███ ░███ ░███░███  ███ ░███ ░███  ███░░███  ░███ ███
     ████ █████░░████████ ████ █████░░██████ ░░██████  ████ █████░░███████  ░░█████
    ░░░░ ░░░░░  ░░░░░░░░ ░░░░ ░░░░░  ░░░░░░   ░░░░░░  ░░░░ ░░░░░  ░░░░░░░░   ░░░░░
    
Autodetected device type: cuda
GPU: NVIDIA GeForce RTX 3050 Ti Laptop GPU | Peak FLOPS (BF16): inf
COMPUTE_DTYPE: torch.bfloat16 (auto-detected: CUDA SM 86 (bf16 supported))


wandb: Currently logged in as: seqaeon (seqaeon-) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!
!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!
Vocab size: 32,768
Model config:
{
  "sequence_len": 32,
  "vocab_size": 32768,
  "n_layer": 4,
  "n_head": 16,
  "n_kv_head": 16,
  "n_embd": 256,
  "use_moe": false,
  "use_perm": false,
  "moe_num_experts": 8,
  "moe_router_dim": 64,
  "moe_embed_dim": 64,
  "moe_scale": 1.0,
  "use_remix_linear": false,
  "remix_context_dim": 64,
  "remix_basis_size": 64,
  "window_pattern": "SSSL"
}


/home/seqaeon/Downloads/nanochat/nanochat/tokenizer.py:405: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  token_bytes = torch.load(f, map_location=device)


Parameter counts:
wte                     : 8,388,608
value_embeds            : 16,777,216
lm_head                 : 8,388,608
transformer_matrices    : 3,146,752
scalars                 : 8
total                   : 36,701,192
Estimated FLOPs per token: 6.945792e+07
Scaling LRs by 0.1250 for batch size 8,192 (reference: 524,288)
Scaling weight decay from 0.200000 to 0.238635 for depth 4
Scaling the LR for the AdamW parameters ∝1/√(256/768) = 1.732051
Using user-provided number of iterations: 20
Total number of training tokens: 163,840
Tokens : Scaling params ratio: 0.01
Total training FLOPs estimate: 1.137999e+13
Tokens / micro-batch / rank: 8 x 32 = 256
Tokens / micro-batch: 256
Total batch size 8,192 => gradient accumulation steps: 32
Step 00000 | Validation bpb: 3.319357
step 00000/00020 (0.00%) | loss: 10.397788 | lrm: 1.00 | dt: 7167.15ms | tok/sec: 1,142 | bf16_mfu: 0.00 | epoch: 1 pq: 0 rg: 1 | total time: 0.00m
step 00001/00020 (5.00%) | loss: 10.386087 | lrm: 1.00 | dt: 1480.

AssertionError: Sequence length grew beyond the rotary embeddings cache: 384 > 320

In [ ]:
BASE_EVAL_CONFIG = {
    'device_type': TRAINING_CONFIG['device_type'],
    'model_tag': TRAINING_CONFIG['model_tag'],
    'step': None,  # None = latest checkpoint
    'eval': 'core,bpb,sample',
    'max_per_task': 500,
    'device_batch_size': 16,
    'split_tokens': 8 * 524288,
}

def base_eval_argv(cfg: dict) -> list[str]:
    flag_map = {
        'device_type': '--device-type',
        'model_tag': '--model-tag',
        'step': '--step',
        'eval': '--eval',
        'max_per_task': '--max-per-task',
        'device_batch_size': '--device-batch-size',
        'split_tokens': '--split-tokens',
    }
    argv = ['scripts.base_eval']
    for k, flag in flag_map.items():
        if k not in cfg or cfg[k] is None:
            continue
        argv.extend([flag, str(cfg[k])])
    return argv

eval_argv = base_eval_argv(BASE_EVAL_CONFIG)
print(' '.join(eval_argv))

In [13]:
# Uncomment to run explicit post-training eval:
import runpy, sys
old_argv = sys.argv[:]
sys.argv = eval_argv
try:
    runpy.run_module('scripts.base_eval', run_name='__main__')
finally:
    sys.argv = old_argv

<frozen runpy>:128: RuntimeWarning: 'scripts.base_eval' found in sys.modules after import of package 'scripts', but prior to execution of 'scripts.base_eval'; this may result in unpredictable behaviour
2026-03-08 17:24:22,134 - nanochat.common - INFO - Distributed world size: 1
2026-03-08 17:24:22,135 - nanochat.checkpoint_manager - INFO - Loading model from /home/seqaeon/.cache/nanochat/base_checkpoints/nb_d20 with step 20
/home/seqaeon/Downloads/nanochat/nanochat/checkpoint_manager.py:64: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbit

Autodetected device type: cuda
Evaluating model: base_model (step 20)
Eval modes: bpb, core, sample

Model Samples

Conditioned samples:
--------------------------------------------------------------------------------
<|bos|>The capital of France is the   20 20 20 20 20 20 
--------------------------------------------------------------------------------
<|bos|>The chemical symbol of gold is the  20 20 20 20 20 20 20
--------------------------------------------------------------------------------
<|bos|>If yesterday was Friday, then tomorrow will be the  20 20 20 20 20 20 20
--------------------------------------------------------------------------------
<|bos|>The opposite of hot is the   20 20 20 20 20 20 
--------------------------------------------------------------------------------
<|bos|>The planets of the solar system are:   20 20 20 20 20 20 20
--------------------------------------------------------------------------------
<|bos|>My favorite color is the  20 20 20 20 20 20 20


OutOfMemoryError: CUDA out of memory. Tried to allocate 1016.00 MiB. GPU 0 has a total capacity of 3.68 GiB of which 916.38 MiB is free. Including non-PyTorch memory, this process has 2.76 GiB memory in use. Of the allocated memory 2.62 GiB is allocated by PyTorch, and 29.06 MiB is reserved by PyTorch but unallocated. If reserved but unallocated memory is large try setting PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True to avoid fragmentation.  See documentation for Memory Management  (https://pytorch.org/docs/stable/notes/cuda.html#environment-variables)

## Notes
- For multi-GPU distributed training from notebook environments, it is usually cleaner to run `torchrun` from a terminal.
- For small smoke tests on CPU/MPS, reduce `depth`, `max_seq_len`, `device_batch_size`, and `training_tokens`.
- `training_tokens` is the main token-budget knob in this notebook.